# Lekcja 13 - Pamięć agenta z wykorzystaniem Knowledge Graphs Cognee


## Konfiguracja

Ten notatnik pokazuje, jak zbudować inteligentnego **asystenta kodowania** z trwałą pamięcią, korzystając z grafów wiedzy [**Cognee**](https://www.cognee.ai/) oraz **Microsoft Agent Framework** (MAF).

Cognee przekształca nieustrukturyzowany tekst w ustrukturyzowany, możliwy do zapytań graf wiedzy oparty na wektorowych osadzeniach — dając twojemu agentowi bogatą, świadomą relacji długoterminową pamięć.

### Czego się nauczysz
1. **Budować Grafy Wiedzy**: Przekształcać profile deweloperów i najlepsze praktyki w ustrukturyzowaną, możliwą do zapytań wiedzę.
2. **Integracja Cognee z MAF**: Używać funkcji `@tool`, aby agent MAF mógł zapytywać graf wiedzy Cognee.
3. **Konwersacje Świadome Sesji**: Utrzymywać kontekst w trakcie wielu pytań w tej samej sesji.
4. **Długoterminowa Pamięć**: Zachowywać ważną wiedzę przez sesje i pobierać ją w nowych rozmowach.

### Wymagania wstępne
- Python 3.9+
- Redis działający lokalnie (`docker run -d -p 6379:6379 redis`) do zarządzania sesjami
- Klucz API LLM (np. OpenAI) — ustaw `LLM_API_KEY` w `.env`
- `CACHING=true` w `.env` (wymagane dla sesji Cognee)
- Projekt Azure AI Foundry z wdrożonym modelem czatu
- `AZURE_AI_PROJECT_ENDPOINT` oraz `AZURE_AI_MODEL_DEPLOYMENT_NAME` w `.env`
- Zalogowany Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Typy pamięci agenta

Ten notatnik bada te same trzy typy pamięci z głównego notatnika Lekcji 13, ale używa Cognee jako zaplecza dla pamięci długoterminowej:

| Typ pamięci | Mechanizm | Czas życia |
|---|---|---|
| **Robocza** | `agent.create_session()` (MAF) | Pojedyncza rozmowa |
| **Krótkoterminowa** | Pamięć sesyjna Cognee (Redis) | Pojedyncza sesja |
| **Długoterminowa** | Graf wiedzy Cognee + wektory | Stała |

### Architektura pamięci Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Przygotuj magazyn Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Część 1 — Budowanie Bazy Wiedzy

Przetwarzamy trzy typy danych, aby stworzyć kompleksową bazę wiedzy dla naszego asystenta programistycznego:

1. **Profil Developera** — osobista wiedza i doświadczenie techniczne
2. **Najlepsze praktyki Pythona** — Zen Pythona z praktycznymi wskazówkami
3. **Historyczne rozmowy** — wcześniejsze sesje pytań i odpowiedzi między programistami a asystentami AI


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Zwizualizuj Graf Wiedzy

Cognee może wygenerować interaktywną wizualizację HTML jednostek i powiązań, które wyodrębnił.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Wzbogacanie pamięci za pomocą Memify

`memify()` analizuje graf wiedzy i generuje inteligentne reguły — identyfikując wzorce, najlepsze praktyki oraz relacje między pojęciami.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Część 2 — Agent MAF z narzędziami Cognee

Teraz tworzymy agenta MAF, który może zapytać graf wiedzy Cognee za pomocą funkcji `@tool`. Pozwala to agentowi wykorzystać pełną moc semantycznego wyszukiwania świadomego grafu, jednocześnie utrzymując kontekst konwersacji przez sesje.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Pamięć robocza z sesjami

`AgentSession` (utworzona za pomocą `agent.create_session()`) zapewnia pamięć roboczą w ramach sesji. Agent może odwoływać się do wcześniejszych wiadomości, jednocześnie zapytując o długoterminowy graf wiedzy Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nowa sesja — pamięć długoterminowa jest zachowana

Rozpoczęcie nowej sesji czyści pamięć roboczą, ale graf wiedzy Cognee jest nadal dostępny. Agent może uzyskać dostęp do tej samej wiedzy długoterminowej w całkowicie nowej rozmowie.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Podsumowanie

W tym notatniku zbudowałeś asystenta kodowania, który łączy **pamięć roboczą MAF** (`agent.create_session()`) z **długoterminową bazą wiedzy Cognee**.

### Czego się nauczyłeś
1. **Budowa grafu wiedzy**: Cognee przetwarza nieustrukturyzowany tekst i tworzy graf + wektorową pamięć.
2. **Wzbogacenie grafu za pomocą memify**: Pochodne fakty i bogatsze relacje na bazie istniejącego grafu.
3. **Integracja MAF + Cognee**: Funkcje `@tool` pozwalają agentowi MAF naturalnie zapytywać graf Cognee.
4. **Pamięć robocza + pamięć długoterminowa**: `AgentSession` (poprzez `agent.create_session()`) zapewnia kontekst sesji, podczas gdy Cognee dostarcza trwałej wiedzy.
5. **Filtrowane wyszukiwanie z NodeSets**: Docieraj do określonych podzbiorów grafu wiedzy (np. tylko zasady).

### Kluczowe wnioski
- **Cognee** przekształca surowy tekst w uporządkowaną, świadomą relacji pamięć — mocniejszą niż zwykły sklep wektorowy.
- Funkcje **`@tool`** łączą agentów MAF z zewnętrznymi systemami wiedzy w przejrzysty sposób.
- **`AgentSession`** (poprzez `agent.create_session()`) oddziela kontekst konkretnej rozmowy od wiedzy długotrwałej.
- Ten sam graf wiedzy obsługuje wiele sesji i agentów.

### Zastosowania w praktyce
- **Kopiloty deweloperskie**: Przegląd kodu, analiza incydentów, asystenci architektury
- **Kopiloty dla klientów**: Agenci wsparcia bazujący na dokumentacji produktowej, FAQ i notatkach CRM
- **Kopiloty wewnętrznych ekspertów**: Asystenci od polityki, prawni, bezpieczeństwa rozumiejący wytyczne
- **Zunifikowane warstwy danych**: Łączenie danych strukturyzowanych i nieustrukturyzowanych w jeden graf zapytaniowy

### Kolejne kroki
- Eksperymentuj ze świadomością czasową w Cognee
- Zdefiniuj ontologię OWL dla jakości domenowego grafu
- Dodaj pętle informacji zwrotnej od użytkowników, aby poprawić wyszukiwanie w czasie
- Skaluj do systemów wieloagentowych współdzielących tę samą warstwę pamięci Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:  
Dokument ten został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mimo naszych starań o dokładność, prosimy mieć na uwadze, że tłumaczenia automatyczne mogą zawierać błędy lub niedokładności. Oryginalny dokument w języku źródłowym powinien być uważany za autorytatywne źródło. W przypadku informacji krytycznych zaleca się skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z korzystania z tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
